# 01 · Spanish industrial voice dataset
Run manually in Google Colab. No API keys or external text-generation service.
Labels originate from semantic records, not predictions of the parser being tested.
Synthetic text is not evidence of noisy-microphone performance. Collect real recordings later.
Families are reserved by split; canonical text is globally deduplicated.


In [ ]:
# Change these values BEFORE running subsequent cells.
USE_DRIVE = True
ROOT = '/content/drive/MyDrive/Robotekin2026'
SEED = 42
DATASET_PATH = ''  # Existing dataset folder on Drive; empty opens upload.
MODEL_PATH = ''    # Existing exported model folder; empty opens upload.
RESUME_RUN = ''    # Existing run folder for checkpoint continuation.
N_EXAMPLES = 25000


In [ ]:
import os, random, subprocess, sys, uuid
from datetime import datetime, timezone
from pathlib import Path
print('Python:', sys.version, '| executable:', sys.executable, flush=True)
install_log = Path('/content/roboqc_pip_install.log')
install_log.parent.mkdir(parents=True, exist_ok=True)
command = [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', 'numpy>=1.26,<3', 'scikit-learn>=1.4,<2', 'joblib>=1.3,<2', 'matplotlib>=3.8,<4', 'pandas>=2.2,<4', 'psutil>=5.9,<8', 'torch>=2.5,<3']
with install_log.open('w', encoding='utf-8') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
        log.write(line)
    exit_code = process.wait()
if exit_code:
    raise RuntimeError(f'pip failed (exit {exit_code}). Read the ERROR lines above. Full log: {install_log}')
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    ROOT = '/content/Robotekin2026'
RUN = Path(RESUME_RUN) if RESUME_RUN else Path(ROOT) / (datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6])
RUN.mkdir(parents=True, exist_ok=bool(RESUME_RUN))
import numpy as np, torch
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
GPU = torch.cuda.is_available()
DEVICE = 'cuda' if GPU else 'cpu'
print('Device:', DEVICE, '| Run:', RUN)
(RUN / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))


In [ ]:
import sys, json, zipfile, shutil, uuid
from pathlib import Path
RUNTIME = Path('/content/roboqc_runtime')
SOURCES = {'roboqc/__init__.py': '', 'roboqc/intent/__init__.py': '', 'roboqc/industrial/__init__.py': '"""Industrial sorting contracts shared by Colab and the local demo."""\n', 'roboqc/industrial/voice.py': '"""Conservative Spanish sorting parser and explicit clarification dialogue.\n\nNo model can bypass relation validation. No robot or GUI dependencies.\n"""\n\nfrom __future__ import annotations\n\nimport copy\nimport re\nimport unicodedata\n\nCOLORS = ("red", "green", "blue")\nCATEGORIES = {"quality": ("good", "defect_1", "defect_2"), "color": COLORS}\nCOLOR_WORDS = {"red": "roja", "green": "verde", "blue": "azul"}\nQUALITY = {\n    "good": r"\\b(?:buen[oa]s?|correct[oa]s?|aceptables?|sin defectos?)\\b",\n    "defect_1": r"\\b(?:defect(?:uos[oa]s?|os?)\\s*(?:de\\s+)?(?:tipo\\s*)?(?:uno|1)|tipo\\s+(?:uno|1))\\b",\n    "defect_2": r"\\b(?:defect(?:uos[oa]s?|os?)\\s*(?:de\\s+)?(?:tipo\\s*)?(?:dos|2)|tipo\\s+(?:dos|2))\\b",\n}\nCOLOR_PATTERNS = {"red": r"\\broj[oa]s?\\b", "green": r"\\bverdes?\\b", "blue": r"\\bazul(?:es)?\\b"}\nDEST = re.compile(r"\\b(?:en|a|al|hacia|dentro de)\\s+(?:(?:la|el|las|los|una?|del)\\s+)?(?:(?:cajas?|contenedor(?:es)?|recipientes?|destinos?)\\s+)?(roj[oa]s?|verdes?|azul(?:es)?|amarill[oa]s?|negr[oa]s?|blanc[oa]s?)\\b")\nSORT = r"\\b(?:clasifica\\w*|ordena\\w*|separa\\w*|distribu\\w*|reparte\\w*|pon|poner|coloca\\w*|deja\\w*|lleva\\w*|envia\\w*|manda\\w*|recoge\\w*|recoger|coge\\w*|recoger|organiza\\w*)\\b"\n\n\ndef normalize(text):\n    return " ".join("".join(c for c in unicodedata.normalize("NFKD", text.casefold()) if not unicodedata.combining(c)).split())\n\n\ndef handles(text):\n    t = normalize(text)\n    if re.match(r"^(?:cuant|donde|muestra|selecciona|inspecciona|hay |que )", t.lstrip("¿")):\n        return False\n    return bool(re.search(r"\\b(?:clasifica\\w*|continua|reanuda|deten|cancela|anula|para|pausa|interrumpe|sigue)\\b", t)\n                or (re.search(SORT, t) and re.search(r"\\b(?:las|piezas|tornillos|tuercas|arandelas|soportes)\\b", t)))\n\n\ndef mentions(text, patterns):\n    return [(m.start(), m.end(), key) for key, pat in patterns.items() for m in re.finditer(pat, text)]\n\n\ndef result(command=None, status="valid", issues=None, clarification=None):\n    return {"schema_version": "industrial-1.0", "status": status, "command": command,\n            "issues": issues or [], "clarification": clarification}\n\n\ndef canonical(command):\n    """Ordering-independent semantic comparison; excludes latency/confidence."""\n    value = copy.deepcopy(command)\n    if value and "rules" in value:\n        value["rules"] = sorted(value["rules"], key=lambda r: (r["category"], r.get("destination") or ""))\n    return value\n\n\ndef understand(text, model=None):\n    t = normalize(text)\n    if not t:\n        return result(status="invalid", issues=["empty_text"])\n    if len(t) > 2000:\n        return result(status="unsupported", issues=["command_too_long"])\n    if re.search(r"(?<!\\w)-\\d+\\b", t):\n        return result(status="invalid", issues=["negative_quantity"])\n    if re.search(r"\\b(?:suelda\\w*|pinta\\w*|taladra\\w*|pule|atornilla\\w*|empaqueta\\w*|desmonta\\w*)\\b", t):\n        return result(status="unsupported", issues=["unsupported_action"])\n    # Match full control phrases: a negated stop must never become a stop.\n    clean = re.sub(r"^(?:por favor[, ]+|robot[, ]+)", "", t).strip(" .!?¿¡")\n    for intent, pattern in {\n        "stop": r"(?:deten|para|interrumpe|pausa)(?:\\s+(?:la\\s+)?(?:clasificacion|proceso|robot))?(?:\\s+por favor)?",\n        "resume": r"(?:continua|reanuda|sigue)(?:\\s+(?:con\\s+)?(?:la\\s+)?(?:clasificacion|clasificando|proceso))?(?:\\s+por favor)?",\n        "cancel": r"(?:cancela|anula)(?:\\s+(?:la\\s+)?(?:orden|peticion|clasificacion))?",\n    }.items():\n        if re.fullmatch(pattern, clean):\n            return result({"intent": intent})\n    if re.search(r"\\b(?:no|nunca)\\b", t) and re.search(r"\\b(?:detengas|pares|continues|reanudes)\\b", t):\n        return result(status="unsupported", issues=["negated_control"])\n    if re.search(r"\\b(?:deten|pausa|continua|reanuda|cancela)\\b|\\bpara\\s+(?:el\\s+(?:proceso|robot)|la\\s+clasificacion)\\b", t) and re.search(SORT, t):\n        return result(status="contradictory", issues=["mixed_control_and_sort"])\n    q = mentions(t, QUALITY)\n    predicted = model.predict(t) if model is not None else None\n    # A learned action can extend verb coverage, but never supply missing destinations.\n    is_sort = bool(re.search(SORT, t)) or bool(predicted and predicted.get("intent") == "sort")\n    if not is_sort:\n        return result(status="unsupported", issues=["unrecognized_action"])\n    if re.search(r"\\b(?:excepto|menos|salvo|no|nunca)\\b", t):\n        return result(status="needs_clarification", issues=["negation_scope"],\n                      clarification="Indica las categorias que quieres clasificar y sus destinos de forma positiva.")\n    destinations = list(DEST.finditer(t))\n    source_text = DEST.sub(" ", t)\n    c = mentions(source_text, COLOR_PATTERNS)\n    if q and c:\n        return result(status="needs_clarification", issues=["mixed_modes"],\n                      clarification="¿Quieres clasificar por calidad o por color?")\n    mode = "quality" if q or re.search(r"\\b(?:defectuosas?|defectuosos?|calidad)\\b", t) else "color"\n    patterns = QUALITY if mode == "quality" else COLOR_PATTERNS\n    rules = []\n    cursor = 0\n    for dest in destinations:\n        clause = t[cursor:dest.start()]\n        categories = [item[2] for item in sorted(mentions(clause, patterns))]\n        consumed_end = dest.end()\n        if not categories:\n            after = re.split(r"\\s+y\\s+|[,;]", t[dest.end():], maxsplit=1)[0]\n            categories = [item[2] for item in sorted(mentions(after, patterns))]\n            if categories:\n                consumed_end += len(after)\n        if mode == "quality" and not categories and re.search(r"\\bdefectuos[oa]s?\\b", clause):\n            categories = ["defect_1", "defect_2"]\n        color = next((key for key, pat in COLOR_PATTERNS.items() if re.search(pat, dest.group(1))), None)\n        if color is None:\n            return result(status="unsupported", issues=["unsupported_destination"])\n        if not categories:\n            return result(status="needs_clarification", issues=["missing_category"],\n                          clarification="¿Que categoria va en la caja indicada?")\n        rules.extend({"category": category, "destination": color} for category in categories)\n        cursor = consumed_end\n    trailing = [item[2] for item in mentions(t[cursor:], patterns)]\n    if destinations and re.search(r"\\b(?:en|hacia|caja|contenedor)\\b", t[cursor:]):\n        return result(status="needs_clarification", issues=["unresolved_trailing_clause"],\n                      clarification="Repite cada categoria con un unico destino completo.")\n    if not destinations:\n        trailing = [item[2] for item in mentions(t, patterns)]\n        if mode == "quality" and not trailing and re.search(r"\\bdefectuos[oa]s?\\b", t):\n            trailing = ["defect_1", "defect_2"]\n    rules.extend({"category": category, "destination": None} for category in trailing)\n    command = {"intent": "sort", "mode": mode, "rules": rules}\n    if re.search(r"\\b(?:solamente|solo|unicamente)\\b", t):\n        command["filter"] = list(dict.fromkeys(r["category"] for r in rules))\n    return validate(command)\n\n\ndef validate(command):\n    command = copy.deepcopy(command)\n    mode = command.get("mode")\n    if mode not in CATEGORIES:\n        return result(status="invalid", issues=["invalid_mode"])\n    unique = {}\n    for rule in command.get("rules", []):\n        cat, dest = rule.get("category"), rule.get("destination")\n        if cat not in CATEGORIES[mode] or (dest is not None and dest not in COLORS):\n            return result(status="unsupported", issues=["unsupported_rule"])\n        if cat in unique and unique[cat] != dest:\n            return result(command, "contradictory", ["conflicting_destinations"])\n        unique[cat] = dest\n    command["rules"] = [{"category": c, "destination": d} for c, d in unique.items()]\n    if not unique:\n        return result(command, "needs_clarification", ["missing_category"], "¿Que categorias y destinos quieres usar?")\n    if any(d is None for d in unique.values()):\n        missing = ", ".join(c for c, d in unique.items() if d is None)\n        return result(command, "needs_clarification", ["missing_destination"],\n                      f"¿A que caja (roja, verde o azul) envio {missing}?")\n    return result(command)\n\n\nclass Dialogue:\n    """A missing destination always needs an explicit answer, even with active rules."""\n\n    def __init__(self):\n        self.pending = None\n        self.active = None\n        self.running = False\n\n    def submit(self, text, model=None):\n        t = normalize(text)\n        if self.pending and not re.search(SORT, t) and not re.search(r"\\b(?:deten|para|pausa|cancela|anula|continua|reanuda)\\b", t):\n            pending_command = self.pending.get("command")\n            if pending_command and "missing_destination" in self.pending["issues"]:\n                candidate = copy.deepcopy(pending_command)\n                explicit = understand("pon " + text)\n                if explicit["status"] == "valid" and explicit.get("command", {}).get("rules"):\n                    updates = {r["category"]: r["destination"] for r in explicit["command"]["rules"]}\n                    for rule in candidate["rules"]:\n                        if rule["destination"] is None and rule["category"] in updates:\n                            rule["destination"] = updates[rule["category"]]\n                else:\n                    colors = {m[2] for m in mentions(t, COLOR_PATTERNS)}\n                    # Only a bare destination answer can apply one color to all missing rules.\n                    if len(colors) == 1 and re.fullmatch(r"(?:(?:a|en)\\s+)?(?:(?:la|el)\\s+)?(?:(?:caja|contenedor)\\s+)?(?:roja?|rojo|verde|azul)[.! ]*", t):\n                        for rule in candidate["rules"]:\n                            if rule["destination"] is None:\n                                rule["destination"] = next(iter(colors))\n                response = validate(candidate)\n            else:\n                response = result(status="needs_clarification", issues=["repeat_complete_command"],\n                                  clarification="Reformula la instruccion completa con categorias y destinos.")\n        else:\n            response = understand(text, model)\n        if response["status"] == "needs_clarification":\n            self.pending = response\n            self.running = False\n        elif response["status"] == "valid":\n            intent = response["command"]["intent"]\n            if intent == "sort":\n                self.active = response["command"]\n                self.pending = None\n                self.running = True\n            elif intent == "cancel":\n                self.pending = None\n                self.active = None\n                self.running = False\n            elif intent == "stop":\n                self.running = False\n            elif intent == "resume":\n                if self.active is None or self.pending:\n                    return result(status="needs_clarification", issues=["no_resumable_plan"],\n                                  clarification="Completa una orden de clasificacion antes de continuar.")\n                self.running = True\n        else:\n            self.running = False\n        return response\n\n\ndef to_legacy(envelope, text, source="text"):\n    from roboqc.commands.schema import Action, StructuredCommand\n    status = {"valid": "VALID", "needs_clarification": "AMBIGUOUS", "unsupported": "UNKNOWN"}.get(envelope["status"], "INVALID")\n    cmd = StructuredCommand(source=source, transcription=text, normalized_text=normalize(text), status=status)\n    cmd.actions = [Action("INDUSTRIAL_COMMAND", {"industrial": envelope})]\n    cmd.validation_errors = list(envelope["issues"])\n    if envelope["clarification"]:\n        cmd.unresolved_references = [envelope["clarification"]]\n    return cmd\n', 'roboqc/industrial/dataset.py': '"""Semantic-first synthetic commands. Called manually from Colab, never on import."""\n\nimport hashlib\nimport itertools\nimport json\nimport random\nfrom pathlib import Path\n\nfrom .voice import CATEGORIES, COLOR_WORDS, normalize, result\n\nQUALITY_WORDS = {\n    "good": ["buenas", "correctas", "aceptables", "sin defectos"],\n    "defect_1": ["defectuosas tipo uno", "con defecto tipo 1", "tipo uno", "defectuosas de tipo uno"],\n    "defect_2": ["defectuosas tipo dos", "con defecto tipo 2", "tipo dos", "defectuosas de tipo dos"],\n}\nPARTS = ["piezas", "tuercas", "arandelas", "tornillos", "soportes", "componentes"]\nVERBS = ["Pon", "Coloca", "Clasifica", "Lleva", "Envia", "Distribuye", "Manda", "Deja", "Separa", "Reparte"]\n# Entire grammatical families belong to one split; variants never cross it.\nFAMILIES = {\n    "train": ["{v} {s} en la caja {d}", "{v} {s} al contenedor {d}", "{v} {s} hacia el recipiente {d}", "{v} {s} dentro de la caja {d}"],\n    "validation": ["{v} a la caja {d}: {s}", "{v} {s} a la {d}"],\n    "test": ["{v} {s} en el destino {d}", "{v} hacia la caja {d} {s}"],\n}\nPREFIXES = ["", "Por favor, ", "Robot, ", "Ahora, ", "Cuando puedas, ", "Para esta tanda, "]\nSUFFIXES = ["", ", por favor", " con cuidado", " despacio", " ahora", " para esta tanda"]\n\n\ndef key(text):\n    return " ".join(__import__("re").sub(r"[^\\w\\s]", " ", normalize(text)).split())\n\n\ndef _render(family, verb, subject, destination):\n    color = "rojo" if destination == "red" and any(word in family for word in ("contenedor", "recipiente", "destino")) else COLOR_WORDS[destination]\n    return family.format(v=verb, s=subject, d=color)\n\n\ndef _subject(category, mode, rng):\n    description = rng.choice(QUALITY_WORDS[category]) if mode == "quality" else {"red":"rojas","green":"verdes","blue":"azules"}[category]\n    part = rng.choice(PARTS)\n    masculine = part in ("tornillos","soportes","componentes")\n    if masculine:\n        for feminine,male in [("buenas","buenos"),("correctas","correctos"),("defectuosas","defectuosos"),("rojas","rojos")]:\n            description = description.replace(feminine,male)\n    return ("los " if masculine else "las ") + part + " " + description\n\n\ndef generate(n=25000, seed=42):\n    if n < 20000:\n        raise ValueError("The voice dataset requires at least 20,000 examples.")\n    rng = random.Random(seed)\n    splits, seen = {}, set()\n    for split, count in zip(FAMILIES, [int(n * .70), int(n * .15), n - int(n * .70) - int(n * .15)]):\n        records = []\n        attempts = 0\n        while len(records) < count:\n            attempts += 1\n            if attempts > count * 300:\n                raise RuntimeError("Unique example space exhausted; expand grammatical families.")\n            mode = rng.choice(list(CATEGORIES))\n            cats = rng.sample(list(CATEGORIES[mode]), rng.randint(1, 3))\n            family_index = rng.randrange(len(FAMILIES[split]))\n            family = FAMILIES[split][family_index]\n            kind = rng.choices(["valid", "missing", "conflict", "unsupported", "control", "invalid"], [.52, .16, .10, .08, .08, .06])[0]\n            clauses, rules = [], []\n            for i, cat in enumerate(cats):\n                subject = _subject(cat,mode,rng)\n                dest = rng.choice(list(COLOR_WORDS))\n                clauses.append(_render(family, rng.choice(VERBS), subject, dest))\n                rules.append({"category": cat, "destination": dest})\n            text = rng.choice(PREFIXES) + " y ".join(clauses) + rng.choice(SUFFIXES)\n            command = {"intent": "sort", "mode": mode, "rules": rules}\n            expected = result(command)\n            if kind == "invalid":\n                text = rng.choice(PREFIXES) + _render(family,rng.choice(VERBS),f"-{rng.randint(1, 99)} piezas buenas",rng.choice(list(COLOR_WORDS)))\n                expected = result(status="invalid", issues=["negative_quantity"])\n            elif kind == "missing":\n                # Generate independently labelled missing-destination instructions.\n                cat = cats[0]\n                subject = _subject(cat,mode,rng)\n                missing_families = {"train":["{v} solamente {s}","{v} solo {s}"],\n                    "validation":["{v} unicamente {s} para esta tanda"],\n                    "test":["{v} para esta tanda solamente {s}"]}\n                text = rng.choice(PREFIXES) + rng.choice(missing_families[split]).format(v=rng.choice(VERBS),s=subject) + rng.choice(SUFFIXES)\n                command = {"intent": "sort", "mode": mode, "rules": [{"category": cat, "destination": None}], "filter": [cat]}\n                expected = result(command, "needs_clarification", ["missing_destination"])\n            elif kind == "conflict":\n                cat = cats[0]\n                other = rng.choice([d for d in COLOR_WORDS if d != rules[0]["destination"]])\n                desc = rng.choice(QUALITY_WORDS[cat]) if mode == "quality" else {"red": "rojas", "green": "verdes", "blue": "azules"}[cat]\n                text += " y " + _render(family, rng.choice(VERBS), "las piezas " + desc, other)\n                command["rules"].append({"category": cat, "destination": other})\n                expected = result(command, "contradictory", ["conflicting_destinations"])\n            elif kind == "unsupported":\n                text = rng.choice(PREFIXES) + _render(family,rng.choice(["Suelda", "Pinta", "Taladra", "Pule", "Atornilla", "Empaqueta"]),_subject(cats[0],mode,rng),rng.choice(list(COLOR_WORDS))) + rng.choice(SUFFIXES)\n                expected = result(status="unsupported", issues=["unrecognized_action"])\n            elif kind == "control":\n                intent = rng.choice(["stop", "resume", "cancel"])\n                phrase = {\n                    "train": {"stop": ["Deten la clasificacion"], "resume": ["Continua clasificando"], "cancel": ["Cancela la orden"]},\n                    "validation": {"stop": ["Interrumpe el proceso"], "resume": ["Reanuda la clasificacion"], "cancel": ["Anula la peticion"]},\n                    "test": {"stop": ["Para el robot"], "resume": ["Sigue con la clasificacion"], "cancel": ["Cancela la clasificacion"]},\n                }[split]\n                text = rng.choice(["", "Por favor, ", "Robot, "]) + rng.choice(phrase[intent]) + rng.choice(["", ".", "!"])\n                expected = result({"intent": intent})\n            norm = key(text)\n            if norm in seen:\n                continue\n            seen.add(norm)\n            group = f"{split}:{family_index}:{kind}"\n            record = {"id": hashlib.sha256(norm.encode()).hexdigest()[:16], "text": text,\n                      "family": f"{split}:{family_index}", "group": group, "split": split,\n                      "expected": expected, "intent": (expected["command"] or {}).get("intent", "unsupported"),\n                      "mode": (expected["command"] or {}).get("mode", "none")}\n            records.append(record)\n        splits[split] = records\n    return splits\n\n\ndef hypotheses():\n    return [f"La orden pide {action}." for action in ("clasificar piezas", "detener la clasificacion", "continuar clasificando", "cancelar la orden")]\n\n\ndef pairs(record):\n    """Relations and action decisions; neutral is distinct from contradiction."""\n    command = record["expected"]["command"] or {}\n    for action, hypothesis in zip(["sort", "stop", "resume", "cancel"], hypotheses()):\n        yield record["text"], hypothesis, "entailment" if command.get("intent") == action else "neutral"\n    for mode in CATEGORIES:\n        yield record["text"], f"La orden pide clasificar por {\'calidad\' if mode == \'quality\' else \'color\'}.", "entailment" if command.get("mode") == mode else "neutral"\n    yield record["text"], "La orden solicita una accion no admitida.", "entailment" if record["intent"] == "unsupported" else "neutral"\n    for cat, dest in itertools.product(CATEGORIES.get(command.get("mode"), []), COLOR_WORDS):\n        matches = [r for r in command.get("rules", []) if r["category"] == cat]\n        label = "entailment" if any(r["destination"] == dest for r in matches) else "contradiction" if any(r["destination"] for r in matches) else "neutral"\n        yield record["text"], f"El destino de {cat} es la caja {dest}.", label\n\n\ndef export(splits, output):\n    import csv\n    from collections import Counter\n    out = Path(output)\n    out.mkdir(parents=True, exist_ok=False)\n    keys, families, report = {}, {}, {}\n    for split, records in splits.items():\n        keys[split] = {key(r["text"]) for r in records}\n        families[split] = {r["family"] for r in records}\n        assert len(keys[split]) == len(records)\n        for record in records:\n            json.dumps(record["expected"], ensure_ascii=False, allow_nan=False)\n        with (out / f"{split}.jsonl").open("w", encoding="utf-8") as f:\n            for record in records:\n                f.write(json.dumps(record, ensure_ascii=False) + "\\n")\n        with (out / f"{split}.csv").open("w", encoding="utf-8", newline="") as f:\n            writer = csv.DictWriter(f, fieldnames=list(records[0]))\n            writer.writeheader()\n            writer.writerows({**r, "expected": json.dumps(r["expected"], ensure_ascii=False)} for r in records)\n        report[split] = {"size": len(records), "status": dict(Counter(r["expected"]["status"] for r in records)),\n                         "intent": dict(Counter(r["intent"] for r in records))}\n    for a, b in itertools.combinations(keys, 2):\n        assert not keys[a] & keys[b], (a, b, "duplicate text")\n        assert not families[a] & families[b], (a, b, "family leakage")\n    (out / "manifest.json").write_text(json.dumps(report, indent=2), encoding="utf-8")\n    return report\n', 'roboqc/industrial/models.py': '"""Portable intent bundles. Heavy backends are imported only when selected."""\n\nimport json\nfrom pathlib import Path\n\nLABELS = ["sort|quality", "sort|color", "stop|none", "resume|none", "cancel|none", "unsupported|none"]\n\n\ndef decode(label, confidence=1.0):\n    intent, mode = label.split("|")\n    return {"intent": intent, "mode": mode, "confidence": float(confidence)}\n\n\nclass ONNXSentenceEncoder:\n    """Sentence embeddings through ONNX Runtime, independent of Optimum."""\n\n    def __init__(self, path):\n        import onnxruntime as ort\n        from transformers import AutoTokenizer\n        path = Path(path)\n        self.config = json.loads((path / "onnx_config.json").read_text(encoding="utf-8"))\n        self.tokenizer = AutoTokenizer.from_pretrained(path, local_files_only=True)\n        self.session = ort.InferenceSession(str(path / self.config["model_file"]), providers=["CPUExecutionProvider"])\n\n    def encode(self, texts, show_progress_bar=False, batch_size=32):\n        import numpy as np\n        outputs = []\n        for start in range(0,len(texts),batch_size):\n            tokens = self.tokenizer(texts[start:start+batch_size],padding=True,truncation=True,\n                                    max_length=self.config["max_seq_length"],return_tensors="np")\n            inputs = {item.name: tokens[item.name].astype(np.int64) for item in self.session.get_inputs()}\n            outputs.append(self.session.run(["sentence_embedding"],inputs)[0])\n        return np.concatenate(outputs) if outputs else np.empty((0,self.config["embedding_dim"]),dtype=np.float32)\n\n\nclass IntentBundle:\n    def __init__(self, path, kind=None):\n        self.root = Path(path)\n        self.manifest = json.loads((self.root / "manifest.json").read_text(encoding="utf-8"))\n        if self.manifest.get("schema_version") != "industrial-1.0":\n            raise ValueError("Unsupported industrial artifact version")\n        self.kind = kind or self.manifest["selected"]\n        if self.kind not in self.manifest["models"]:\n            raise ValueError(f"Model {self.kind} was not exported")\n        self.threshold = self.manifest["models"][self.kind].get("threshold", 0.0)\n        self._cache = {}\n        if self.kind == "tfidf":\n            import joblib\n            self.model = joblib.load(self.root / "tfidf.joblib")\n        elif self.kind == "minilm":\n            import joblib\n            self.model = joblib.load(self.root / "minilm.joblib")\n            info = self.manifest["models"][self.kind]\n            path = self.root / info.get("encoder_path", "encoder")\n            if info.get("encoder_backend") == "onnx_native":\n                self.encoder = ONNXSentenceEncoder(path)\n            else:\n                from sentence_transformers import SentenceTransformer\n                kwargs = {"backend": info.get("encoder_backend", "torch")}\n                if info.get("encoder_file"):\n                    kwargs["model_kwargs"] = {"file_name": info["encoder_file"]}\n                self.encoder = SentenceTransformer(str(path), device="cpu", **kwargs)\n        elif self.kind == "jepa":\n            import torch\n            from roboqc.intent.jepa_model import CharVocab, JEPAIntentModel\n            ckpt = torch.load(self.root / "jepa.pt", map_location="cpu", weights_only=True)\n            self.vocab = CharVocab.from_dict(ckpt["vocab"])\n            self.labels = ckpt["labels"]\n            self.max_len = ckpt["max_len"]\n            self.model = JEPAIntentModel(len(self.vocab.stoi), len(self.labels))\n            self.model.load_state_dict(ckpt["state_dict"])\n            self.model.eval()\n        elif self.kind == "openjev":\n            from transformers import AutoModelForSequenceClassification, AutoTokenizer\n            self.tokenizer = AutoTokenizer.from_pretrained(self.root / "openjev", local_files_only=True)\n            self.model = AutoModelForSequenceClassification.from_pretrained(self.root / "openjev", local_files_only=True).eval()\n        else:\n            raise ValueError(f"Unknown model: {self.kind}")\n\n    def relation_scores(self, text, hypotheses):\n        import torch\n        template = getattr(self.model.config, "nli_template", None)\n        if not template:\n            raise ValueError("OpenJev artifact must define its pretrained nli_template")\n        encoded = self.tokenizer([template.format(premise=text, hypothesis=h) for h in hypotheses], return_tensors="pt", padding=True, truncation=True, max_length=512)\n        with torch.inference_mode():\n            probs = self.model(**encoded).logits.softmax(-1)\n        mapping = {str(k).lower(): int(v) for k, v in self.model.config.label2id.items()}\n        ent = mapping.get("entailment")\n        if ent is None:\n            raise ValueError("OpenJev artifact is missing the entailment label mapping")\n        return probs[:, ent].cpu().numpy()\n\n    def predict(self, text):\n        if text in self._cache:\n            value = self._cache[text]\n            return decode("unsupported|none",value["confidence"]) if value["confidence"] < self.threshold else value\n        if self.kind in ("tfidf", "minilm"):\n            x = [text] if self.kind == "tfidf" else self.encoder.encode([text], show_progress_bar=False)\n            p = self.model.predict_proba(x)[0]\n            index = p.argmax()\n            value = decode(str(self.model.classes_[index]), p[index])\n        elif self.kind == "jepa":\n            import torch\n            if len(text) > self.max_len:\n                return decode("unsupported|none", 0.0)\n            with torch.inference_mode():\n                p = self.model.classifier(self.model.encode(torch.tensor([self.vocab.encode(text, self.max_len)]))).softmax(-1)[0]\n            value = decode(self.labels[int(p.argmax())], float(p.max()))\n        else:\n            hs = ["La orden pide clasificar por calidad.", "La orden pide clasificar por color.",\n                  "La orden pide detener la clasificacion.", "La orden pide continuar clasificando.",\n                  "La orden pide cancelar la orden.", "La orden solicita una accion no admitida."]\n            p = self.relation_scores(text, hs)\n            index = p.argmax()\n            value = decode(LABELS[index], p[index])\n        if len(self._cache) >= 4096:\n            self._cache.clear()\n        self._cache[text] = value\n        return decode("unsupported|none",value["confidence"]) if value["confidence"] < self.threshold else value\n', 'roboqc/industrial/training.py': '"""Colab training and evaluation helpers. No training runs at import time."""\n\nimport copy\nimport json\nimport random\nimport time\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .models import IntentBundle\nfrom .voice import canonical, understand\n\n\ndef load_records(path):\n    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]\n\n\ndef target(record):\n    return record["intent"] + "|" + record["mode"]\n\n\ndef seed_all(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    import torch\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef export_minilm_onnx(output, sample):\n    """Export the full sentence encoder (including pooling), then validate INT8."""\n    import torch\n    import joblib\n    import onnxruntime as ort\n    from onnxruntime.quantization import QuantType, quantize_dynamic\n    from sentence_transformers import SentenceTransformer\n    from .models import ONNXSentenceEncoder\n    out = Path(output)\n    original = SentenceTransformer(str(out / "encoder"),device="cpu",model_kwargs={"attn_implementation":"eager"})\n    original.eval()\n    destination = out / "encoder_int8"\n    destination.mkdir(exist_ok=True)\n    original.tokenizer.save_pretrained(destination)\n    class Encoder(torch.nn.Module):\n        def __init__(self, model):\n            super().__init__()\n            self.model = model\n\n        def forward(self, input_ids, attention_mask):\n            return self.model({"input_ids":input_ids,"attention_mask":attention_mask})["sentence_embedding"]\n    tokens = original.tokenize(sample[:2])\n    fp32_path = destination / "model.onnx"\n    torch.onnx.export(Encoder(original).eval(),(tokens["input_ids"],tokens["attention_mask"]),str(fp32_path),\n        input_names=["input_ids","attention_mask"],output_names=["sentence_embedding"],opset_version=17,dynamo=False,\n        dynamic_axes={"input_ids":{0:"batch",1:"sequence"},"attention_mask":{0:"batch",1:"sequence"},"sentence_embedding":{0:"batch"}})\n    quantize_dynamic(str(fp32_path),str(destination / "model_int8.onnx"),weight_type=QuantType.QInt8,\n                     op_types_to_quantize=["MatMul"],per_channel=True)\n    config = {"model_file":"model_int8.onnx","max_seq_length":original.max_seq_length,\n              "embedding_dim":original.get_sentence_embedding_dimension()}\n    (destination / "onnx_config.json").write_text(json.dumps(config,indent=2),encoding="utf-8")\n    a = original.encode(sample,show_progress_bar=False)\n    fp32 = ort.InferenceSession(str(fp32_path),providers=["CPUExecutionProvider"])\n    reference_tokens = original.tokenizer(sample,padding=True,truncation=True,max_length=original.max_seq_length,return_tensors="np")\n    b = fp32.run(None,{item.name:reference_tokens[item.name].astype(np.int64) for item in fp32.get_inputs()})[0]\n    np.testing.assert_allclose(a,b,atol=1e-4,rtol=1e-3)\n    quantized = ONNXSentenceEncoder(destination)\n    c = quantized.encode(sample)\n    cosine = np.sum(a*c,axis=1)/(np.linalg.norm(a,axis=1)*np.linalg.norm(c,axis=1))\n    head = joblib.load(out / "minilm.joblib")\n    agreement = float(np.mean(head.predict(a) == head.predict(c)))\n    return {"minimum_embedding_cosine":float(cosine.min()),"classification_agreement":agreement,\n            "passed":bool(cosine.min() >= .99 and agreement == 1.0),"backend":"onnxruntime"}\n\n\ndef train_linear(train, validation, output, encoder_name=None, seed=42, epochs=40, patience=5):\n    import joblib\n    from sklearn.feature_extraction.text import TfidfVectorizer\n    from sklearn.pipeline import FeatureUnion, Pipeline\n    from sklearn.linear_model import SGDClassifier\n    from sklearn.metrics import f1_score\n    out = Path(output)\n    name = "minilm" if encoder_name else "tfidf"\n    if encoder_name:\n        import torch\n        from sentence_transformers import SentenceTransformer\n        encoder = SentenceTransformer(encoder_name, device="cuda" if torch.cuda.is_available() else "cpu")\n        encoder.save(str(out / "encoder"))\n        x = encoder.encode([r["text"] for r in train], batch_size=64, show_progress_bar=True)\n        vx = encoder.encode([r["text"] for r in validation], batch_size=64, show_progress_bar=True)\n        np.save(out / "train_embeddings.npy", x)\n        np.save(out / "validation_embeddings.npy", vx)\n        vectorizer = None\n    else:\n        vectorizer = FeatureUnion([\n            ("word", TfidfVectorizer(ngram_range=(1, 2), max_features=30000)),\n            ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=50000)),\n        ])\n        x = vectorizer.fit_transform([r["text"] for r in train])\n        vx = vectorizer.transform([r["text"] for r in validation])\n    y, vy = np.array([target(r) for r in train]), [target(r) for r in validation]\n    model = SGDClassifier(loss="log_loss", random_state=seed, alpha=1e-5)\n    best, stale, history = -1, 0, []\n    for epoch in range(epochs):\n        order = np.random.default_rng(seed + epoch).permutation(len(y))\n        model.partial_fit(x[order], y[order], classes=np.unique(y))\n        score = f1_score(vy, model.predict(vx), average="macro", zero_division=0)\n        history.append({"epoch": epoch + 1, "macro_f1": float(score)})\n        if score > best:\n            best, stale = score, 0\n            best_model = copy.deepcopy(model)\n            artifact = best_model if vectorizer is None else Pipeline([("features", vectorizer), ("classifier", best_model)])\n            joblib.dump(artifact, out / f"{name}.joblib")\n        else:\n            stale += 1\n        (out / f"{name}_history.json").write_text(json.dumps(history), encoding="utf-8")\n        if stale >= patience:\n            break\n    return {"validation_macro_f1": float(best), "threshold": 0.0}\n\n\ndef train_jepa(train, validation, output, seed=42, epochs=30, patience=5):\n    import torch\n    from roboqc.intent.jepa_model import CharVocab, JEPAIntentModel, jepa_loss\n    from sklearn.metrics import f1_score\n    seed_all(seed)\n    out = Path(output)\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    labels = sorted({target(r) for r in train})\n    vocab = CharVocab()\n    vocab.build([r["text"] for r in train])\n    max_len = 512\n    if any(len(r["text"]) > max_len for r in train + validation):\n        raise ValueError("JEPA max_len would truncate a labelled instruction")\n    x = torch.tensor([vocab.encode(r["text"], max_len) for r in train])\n    y = torch.tensor([labels.index(target(r)) for r in train])\n    vx = torch.tensor([vocab.encode(r["text"], max_len) for r in validation])\n    vy = [target(r) for r in validation]\n    model = JEPAIntentModel(len(vocab.stoi), len(labels)).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)\n    checkpoint = out / "jepa_checkpoint.pt"\n    first, best, stale, history = 0, -1.0, 0, []\n    if checkpoint.exists():\n        state = torch.load(checkpoint, map_location=device, weights_only=True)\n        model.load_state_dict(state["model"])\n        optimizer.load_state_dict(state["optimizer"])\n        first, best, stale, history = state["epoch"], state["best"], state["stale"], state["history"]\n    for epoch in range(first, epochs):\n        model.train()\n        for ids in torch.randperm(len(x)).split(64):\n            full = x[ids].to(device)\n            masked = full.clone()\n            masked[(torch.rand_like(full.float()) < .15) & (full != 0)] = 2\n            optimizer.zero_grad()\n            pred, target_z, logits = model.forward_train(masked, full)\n            loss, _ = jepa_loss(pred, target_z, logits, y[ids].to(device))\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            optimizer.step()\n            model.update_target_encoder()\n        model.eval()\n        predictions = []\n        with torch.inference_mode():\n            for batch in vx.split(128):\n                predictions += [labels[i] for i in model.classifier(model.encode(batch.to(device))).argmax(-1).cpu().tolist()]\n        score = float(f1_score(vy, predictions, average="macro", zero_division=0))\n        history.append({"epoch": epoch + 1, "macro_f1": score})\n        if score > best:\n            best, stale = score, 0\n            torch.save({"state_dict": model.cpu().state_dict(), "vocab": vocab.to_dict(), "labels": labels, "max_len": max_len}, out / "jepa.pt")\n            model.to(device)\n        else:\n            stale += 1\n        torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "epoch": epoch + 1,\n                    "best": best, "stale": stale, "history": history}, checkpoint)\n        (out / "jepa_history.json").write_text(json.dumps(history), encoding="utf-8")\n        if stale >= patience:\n            break\n    return {"validation_macro_f1": best, "threshold": 0.0}\n\n\ndef train_openjev(train, validation, output, epochs=3, seed=42):\n    import torch\n    from datasets import Dataset\n    from peft import LoraConfig, TaskType, get_peft_model\n    from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,\n                              EarlyStoppingCallback, Trainer, TrainingArguments)\n    from sklearn.metrics import f1_score\n    from .dataset import pairs\n    if not torch.cuda.is_available():\n        raise RuntimeError("Enable a Colab GPU for OpenJev LoRA training.")\n    name, sub = "AlexWortega/openjev", "qwen3.5-0.8b-nli-v5"\n    tokenizer = AutoTokenizer.from_pretrained(name, subfolder=sub)\n    if tokenizer.pad_token_id is None:\n        tokenizer.pad_token = tokenizer.eos_token\n    model = AutoModelForSequenceClassification.from_pretrained(name, subfolder=sub, torch_dtype=torch.float32)\n    mapping = {k.lower(): int(v) for k, v in model.config.label2id.items()}\n    if set(mapping) != {"entailment", "contradiction", "neutral"}:\n        raise ValueError(f"Inspect upstream label mapping before training: {mapping}")\n    model.config.pad_token_id = tokenizer.pad_token_id\n    model.config.use_cache = False\n    template = getattr(model.config, "nli_template", None)\n    if not template:\n        raise ValueError("Inspect the pretrained OpenJev nli_template before training")\n    model = get_peft_model(model, LoraConfig(task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16,\n                                           target_modules=["q_proj", "v_proj"], lora_dropout=.05))\n    def prepare(records):\n        rows = [{"premise": p, "hypothesis": h, "labels": mapping[label]} for r in records for p, h, label in pairs(r)]\n        data = Dataset.from_list(rows)\n        return data.map(lambda b: tokenizer([template.format(premise=p, hypothesis=h) for p,h in zip(b["premise"], b["hypothesis"])], truncation=True, max_length=512), batched=True,\n                        remove_columns=["premise", "hypothesis"])\n    out = Path(output)\n    args = TrainingArguments(output_dir=str(out / "openjev_checkpoints"), num_train_epochs=epochs,\n        per_device_train_batch_size=2, per_device_eval_batch_size=4, gradient_accumulation_steps=8,\n        learning_rate=2e-4, eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,\n        metric_for_best_model="macro_f1", save_total_limit=2, seed=seed, report_to="none",\n        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False})\n    trainer = Trainer(model=model, args=args, train_dataset=prepare(train), eval_dataset=prepare(validation),\n        processing_class=tokenizer, data_collator=DataCollatorWithPadding(tokenizer),\n        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],\n        compute_metrics=lambda p: {"macro_f1": f1_score(p.label_ids, p.predictions.argmax(-1), average="macro", zero_division=0)})\n    checkpoints = list((out / "openjev_checkpoints").glob("checkpoint-*"))\n    trainer.train(resume_from_checkpoint=True if checkpoints else None)\n    merged = trainer.model.merge_and_unload()\n    merged.save_pretrained(out / "openjev", safe_serialization=True)\n    tokenizer.save_pretrained(out / "openjev")\n    return {"threshold": 0.0, "upstream": name, "subfolder": sub}\n\n\ndef evaluate(records, model=None, clear_cache=True):\n    from sklearn.metrics import classification_report, confusion_matrix\n    predictions, timings = [], []\n    if model is not None and clear_cache:\n        model._cache.clear()\n    for record in records:\n        start = time.perf_counter()\n        response = understand(record["text"], model)\n        timings.append((time.perf_counter() - start) * 1000)\n        predictions.append(response)\n    true_status = [r["expected"]["status"] for r in records]\n    pred_status = [p["status"] for p in predictions]\n    true_intents = [(r["expected"]["command"] or {}).get("intent", "unsupported") for r in records]\n    pred_intents = [(p["command"] or {}).get("intent", "unsupported") for p in predictions]\n    def fields(command):\n        return {(r["category"], r["destination"]) for r in (command or {}).get("rules", [])}\n    tp = fp = fn = exact = false_accept = valid_count = 0\n    field_correct = {"category": 0, "destination": 0, "mode": 0}\n    entity_counts = {"category":[0,0,0],"destination":[0,0,0]}\n    for record, prediction in zip(records, predictions):\n        gold = record["expected"]\n        equal = prediction["status"] == gold["status"]\n        if gold["status"] in ("valid", "needs_clarification"):\n            equal &= canonical(prediction["command"]) == canonical(gold["command"])\n        exact += equal\n        false_accept += gold["status"] != "valid" and prediction["status"] == "valid"\n        a, b = fields(gold["command"]), fields(prediction["command"])\n        tp += len(a & b); fp += len(b - a); fn += len(a - b)\n        for field, index in [("category", 0), ("destination", 1)]:\n            gold_values,pred_values = {v[index] for v in a},{v[index] for v in b}\n            field_correct[field] += gold_values == pred_values\n            totals = entity_counts[field]\n            totals[0] += len(gold_values & pred_values)\n            totals[1] += len(pred_values-gold_values)\n            totals[2] += len(gold_values-pred_values)\n        field_correct["mode"] += (gold["command"] or {}).get("mode") == (prediction["command"] or {}).get("mode")\n        valid_count += gold["status"] == "valid"\n    labels = sorted(set(true_status + pred_status))\n    report = {"n": len(records), "complete_command_accuracy": exact / max(1, len(records)),\n        "false_acceptance_rate": false_accept / max(1, len(records) - valid_count),\n        "relation_f1": 2 * tp / max(1, 2 * tp + fp + fn),\n        "entity_accuracy": {k: v / max(1, len(records)) for k, v in field_correct.items()},\n        "entity_f1": {k: 2*v[0]/max(1,2*v[0]+v[1]+v[2]) for k,v in entity_counts.items()},\n        "intent": classification_report(true_intents, pred_intents, output_dict=True, zero_division=0),\n        "status": classification_report(true_status, pred_status, output_dict=True, zero_division=0),\n        "confusion_labels": labels, "confusion_matrix": confusion_matrix(true_status, pred_status, labels=labels).tolist(),\n        "latency_ms": {"p50": float(np.percentile(timings, 50)), "p95": float(np.percentile(timings, 95))}}\n    if model is not None:\n        raw = [model.predict(r["text"]) for r in records]\n        report["raw_model_intent"] = classification_report([r["intent"] for r in records], [r["intent"] for r in raw], output_dict=True, zero_division=0)\n        report["raw_model_mode_accuracy"] = float(np.mean([a["mode"] == b["mode"] for a,b in zip(records, raw)]))\n    return report, predictions\n\n\ndef select_and_calibrate(output, validation):\n    """Only validation chooses thresholds and model; test is never consulted."""\n    out = Path(output)\n    manifest = json.loads((out / "manifest.json").read_text())\n    ranks = []\n    for kind in manifest["models"]:\n        model = IntentBundle(out, kind)\n        model.predict(validation[0]["text"])\n        start = time.perf_counter()\n        for record in validation[1:17]:\n            model.predict(record["text"])\n        uncached_latency = (time.perf_counter()-start)*1000 / max(1,len(validation[1:17]))\n        best = None\n        for threshold in [0.0, .5, .7, .85, .95]:\n            model.threshold = threshold\n            report, _ = evaluate(validation, model, clear_cache=False)\n            rank = (report["false_acceptance_rate"], -report["complete_command_accuracy"], -report["raw_model_intent"]["macro avg"]["f1-score"],uncached_latency)\n            if best is None or rank < best[0]:\n                best = rank, threshold, report\n        manifest["models"][kind].update(threshold=best[1], validation=best[2],uncached_cpu_ms=uncached_latency)\n        ranks.append((best[0], kind))\n        del model\n    manifest["selected"] = min(ranks)[1]\n    (out / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")\n    return manifest\n', 'roboqc/intent/jepa_model.py': '"""Compact JEPA-inspired text representation model for Spanish intent classification.\n\nA small character-level encoder predicts a target (full sentence) latent\nrepresentation from a context (partially masked sentence) representation.\nAn EMA target encoder and a variance regulariser are used as anti-collapse\nmeasures. A linear classification head is trained jointly.\n"""\n\nfrom __future__ import annotations\n\nimport math\nfrom typing import Dict, List, Optional\n\ntry:\n    import torch\n    import torch.nn as nn\n    import torch.nn.functional as F\n    _TORCH = True\nexcept Exception:  # pragma: no cover\n    _TORCH = False\n    torch = None\n    nn = None\n    F = None\n\n\ndef is_available() -> bool:\n    return _TORCH\n\n\nPAD_TOKEN = "<pad>"\nUNK_TOKEN = "<unk>"\nMASK_TOKEN = "<mask>"\n\n\nclass CharVocab:\n    def __init__(self):\n        self.stoi: Dict[str, int] = {PAD_TOKEN: 0, UNK_TOKEN: 1, MASK_TOKEN: 2}\n        self.itos: List[str] = [PAD_TOKEN, UNK_TOKEN, MASK_TOKEN]\n\n    def build(self, texts: List[str]) -> None:\n        chars = set()\n        for t in texts:\n            for ch in t.lower():\n                chars.add(ch)\n        for ch in sorted(chars):\n            if ch not in self.stoi:\n                self.stoi[ch] = len(self.itos)\n                self.itos.append(ch)\n\n    def encode(self, text: str, max_len: int = 128) -> List[int]:\n        ids = [self.stoi.get(ch.lower(), self.stoi[UNK_TOKEN]) for ch in text.lower()]\n        ids = ids[:max_len]\n        if len(ids) < max_len:\n            ids = ids + [self.stoi[PAD_TOKEN]] * (max_len - len(ids))\n        return ids\n\n    def mask_sequence(self, ids: List[int], mask_prob: float = 0.15) -> List[int]:\n        out = list(ids)\n        for i in range(len(out)):\n            if out[i] in (0,):  # skip pad\n                continue\n            if torch.rand(1).item() < mask_prob:  # type: ignore\n                out[i] = self.stoi[MASK_TOKEN]\n        return out\n\n    def to_dict(self) -> Dict:\n        return {"stoi": self.stoi, "itos": self.itos}\n\n    @classmethod\n    def from_dict(cls, d: Dict) -> "CharVocab":\n        v = cls()\n        v.stoi = dict(d["stoi"])\n        v.itos = list(d["itos"])\n        return v\n\n\nif _TORCH:\n    class TextEncoder(nn.Module):\n        def __init__(self, vocab_size: int, embed_dim: int = 32, hidden: int = 64, latent: int = 64):\n            super().__init__()\n            self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)\n            self.conv1 = nn.Conv1d(embed_dim, hidden, kernel_size=3, padding=1)\n            self.conv2 = nn.Conv1d(hidden, hidden, kernel_size=3, padding=1)\n            self.proj = nn.Linear(hidden, latent)\n\n        def forward(self, x):\n            # x: (B, L) long\n            e = self.embedding(x).transpose(1, 2)  # (B, E, L)\n            h = torch.relu(self.conv1(e))\n            h = torch.relu(self.conv2(h))\n            h = h.max(dim=2).values  # (B, hidden)\n            z = self.proj(h)\n            return z\n\n    class JEPAIntentModel(nn.Module):\n        def __init__(self, vocab_size: int, num_classes: int, embed_dim: int = 32,\n                     hidden: int = 64, latent: int = 64):\n            super().__init__()\n            self.embed_dim = embed_dim\n            self.hidden = hidden\n            self.latent = latent\n            self.context_encoder = TextEncoder(vocab_size, embed_dim, hidden, latent)\n            self.target_encoder = TextEncoder(vocab_size, embed_dim, hidden, latent)\n            self.predictor = nn.Sequential(\n                nn.Linear(latent, latent),\n                nn.ReLU(),\n                nn.Linear(latent, latent),\n            )\n            self.classifier = nn.Linear(latent, num_classes)\n            # init target encoder = context encoder\n            self.target_encoder.load_state_dict(self.context_encoder.state_dict())\n            for p in self.target_encoder.parameters():\n                p.requires_grad = False\n\n        def encode(self, x):\n            return self.context_encoder(x)\n\n        def forward_train(self, context_ids, target_ids):\n            z_c = self.context_encoder(context_ids)\n            with torch.no_grad():\n                z_t = self.target_encoder(target_ids)\n            z_pred = self.predictor(z_c)\n            logits = self.classifier(z_c)\n            return z_pred, z_t, logits\n\n        @torch.no_grad()\n        def update_target_encoder(self, tau: float = 0.996):\n            for ps, pt in zip(self.context_encoder.parameters(), self.target_encoder.parameters()):\n                pt.data.mul_(tau).add_(ps.data, alpha=1 - tau)\n\n    def variance_loss(z, gamma: float = 1.0):\n        # VICReg-style variance: keep std above gamma\n        std = torch.sqrt(z.var(dim=0) + 1e-4)\n        return torch.mean(torch.relu(gamma - std))\n\n    def jepa_loss(z_pred, z_t, logits, labels, lambda_var: float = 0.1):\n        pred_loss = F.mse_loss(z_pred, z_t)\n        cls_loss = F.cross_entropy(logits, labels)\n        var = variance_loss(z_pred) + variance_loss(z_t.detach())\n        total = pred_loss + cls_loss + lambda_var * var\n        return total, {"pred": pred_loss.item(), "cls": cls_loss.item(), "var": var.item()}\nelse:\n    TextEncoder = None\n    JEPAIntentModel = None\n    def variance_loss(*a, **k):\n        raise RuntimeError("torch no disponible")\n    def jepa_loss(*a, **k):\n        raise RuntimeError("torch no disponible")\n'}
for relative, source in SOURCES.items():
    target = RUNTIME / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding='utf-8')
sys.path.insert(0, str(RUNTIME))
def package_runtime(folder):
    destination = Path(folder) / 'runtime'
    shutil.copytree(RUNTIME / 'roboqc', destination / 'roboqc', dirs_exist_ok=True)
def local_requirements(folder, packages):
    from importlib.metadata import version, PackageNotFoundError
    rows = []
    for package in packages:
        try:
            # Colab CUDA build suffixes are not portable to Windows CPU wheels.
            rows.append(package + '==' + version(package).split('+')[0])
        except PackageNotFoundError:
            pass
    (Path(folder) / 'requirements-local.txt').write_text('\n'.join(rows) + '\n')
def upload_folder():
    from google.colab import files
    uploaded = files.upload()
    target = RUN / ('uploaded_' + uuid.uuid4().hex[:8])
    target.mkdir()
    for name, blob in uploaded.items():
        path = target / Path(name).name
        path.write_bytes(blob)
        if path.suffix == '.zip':
            with zipfile.ZipFile(path) as archive:
                root = target.resolve()
                for member in archive.infolist():
                    if not (root / member.filename).resolve().is_relative_to(root):
                        raise ValueError('Unsafe ZIP member')
                archive.extractall(target)
    return target
def archive_export(folder):
    package_runtime(folder)
    archive = shutil.make_archive(str(folder), 'zip', folder)
    print('Export:', archive)
    return archive


In [ ]:
from roboqc.industrial.dataset import generate, export
splits = generate(N_EXAMPLES, SEED)
DATA = RUN / 'voice_dataset'
report = export(splits, DATA)
(DATA / 'config.json').write_text(json.dumps({'seed': SEED, 'n': N_EXAMPLES, 'split': [0.70, 0.15, 0.15]}, indent=2))
print(json.dumps(report, indent=2))


Inspect class coverage and cross-split near duplicates. Prefix and polite-word variants are stripped for this diagnostic; investigate hits before training.


In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
counts = pd.DataFrame({s: r['status'] for s, r in report.items()}).fillna(0)
display(counts)
counts.plot.bar(title='Status counts by split'); plt.show()
for s, rows in splits.items():
    display(pd.DataFrame(rows).sample(min(5, len(rows)), random_state=SEED)[['text', 'expected']])
    assert {'sort', 'stop', 'resume', 'cancel', 'unsupported'} <= {r['intent'] for r in rows}, s
# Bounded diagnostic, not a claim that synthetic data has no semantic overlap.
texts = {s: [r['text'] for r in rows[:2000]] for s, rows in splits.items()}
vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5))
x = vectorizer.fit_transform(texts['train'])
nn = NearestNeighbors(n_neighbors=1, metric='cosine').fit(x)
near = {}
for split in ['validation', 'test']:
    distances, indices = nn.kneighbors(vectorizer.transform(texts[split]))
    near[split] = [{'text': texts[split][i], 'train_match': texts['train'][int(indices[i, 0])], 'similarity': float(1-d[0])}
                   for i, d in enumerate(distances) if d[0] < .02]
(DATA / 'near_duplicate_audit.json').write_text(json.dumps(near, ensure_ascii=False, indent=2))
print('Near-duplicate audit counts:', {k: len(v) for k, v in near.items()})


In [ ]:
archive = archive_export(DATA)
print('Use this folder/ZIP in notebook 02. CSV and JSONL are both included.')
